1. Baseline Eventlog
↓
2. 3 Blöcke schließen (T22,T23,T24)
↓
3. Demand auf übrige Blöcke verteilen
↓
4. Blockauslastung neu berechnen
↓
5. Scenario Eventlog speichern

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
log = pd.read_csv(
    "s4_eventlog_scenario_block_resource_log_v2.csv"
)

for col in [
    "enabled:timestamp",
    "start:timestamp",
    "time:timestamp"
]:
    log[col] = pd.to_datetime(log[col])

print(log.shape)

In [ ]:
closed_blocks = [
    "T22",
    "T23",
    "T24"
]

print(
    log["block"]
    .value_counts()
    .loc[closed_blocks]
)

In [ ]:
affected_cases = (

    log.loc[
        log["block"].isin(
            closed_blocks
        ),
        "case:concept:name"
    ]

    .unique()

)

print(
    f"Affected cases: "
    f"{len(affected_cases):,}"
)

In [ ]:
available_blocks = sorted(

    set(log["block"])

    -

    set(closed_blocks)

)

print(
    f"Available blocks: "
    f"{len(available_blocks)}"
)

print(available_blocks)

In [ ]:
remaining_probs = (

    log[
        ~log["block"].isin(
            closed_blocks
        )
    ]

    ["block"]

    .value_counts(
        normalize=True
    )

)

remaining_probs

In [ ]:
np.random.seed(42)

mapping = {}

for case_id in affected_cases:

    mapping[case_id] = np.random.choice(

        remaining_probs.index,

        p=remaining_probs.values

    )

In [ ]:
scenario_log = log.copy()

scenario_log["block"] = (

    scenario_log

    .apply(

        lambda row:

        mapping[
            row["case:concept:name"]
        ]

        if row[
            "case:concept:name"
        ] in mapping

        else row["block"],

        axis=1

    )

)

In [ ]:
scenario_log["org:resource"] = (

    scenario_log

    .apply(

        lambda row:

        row["block"]

        if row["concept:name"] in [

            "RMG_receive",

            "RMG_delivery",

            "RMG_mixed"

        ]

        else row["org:resource"],

        axis=1

    )

)

In [ ]:
# Here no T22 -ö T24 should appear 
print(

    scenario_log["block"]

    .value_counts()

    .head(30)

)

In [ ]:
# CALCULATE NEW UTILIZATION
n_before = log["block"].nunique()

n_after = scenario_log[
    "block"
].nunique()

capacity_factor = (
    n_before
    /
    n_after
)

print(
    f"Capacity factor: "
    f"{capacity_factor:.3f}"
)

In [ ]:
scenario_log["block_utilization"] = (

    scenario_log[
        "block_utilization"
    ]

    * capacity_factor

).clip(
    upper=1.0
)

In [ ]:
#CHECK
print(
    log["block_utilization"]
    .describe()
)

print()

print(
    scenario_log[
        "block_utilization"
    ]
    .describe()
)

In [ ]:
# FINAL CHECK
print(
    scenario_log[
        "block"
    ]
    .isin(
        closed_blocks
    )
    .sum()
)

In [ ]:
scenario_log.to_csv(

    "scenario_A_3blocks_closed.csv",

    index=False

)

print(
    "Scenario saved."
)